# Camera-ready remaining GPU experiments — Colab A100

This notebook runs the two unfinished **GPU** jobs from the camera-ready plan:

1. the seed-42 **Gemma 4 12B mechanistic sweep** (13,600 steering rows plus 2,800 patching rows), and
2. the unfinished **Gemma 4 12B long-cap Strong-IP panel** (2,700 sycophancy rows total).

Both jobs are resumable. Their JSONL checkpoints and logs live on Google Drive, so a Colab disconnect only requires rerunning the setup and run cells. The completed Qwen sweep is not rerun here.

Human annotation, judge grading, figure regeneration, and paper editing happen after this notebook and do not require an A100.

## Before running

Use a **Colab A100 High-RAM** runtime. Add a read-scoped `HF_TOKEN` in Colab Secrets with access to the private `ic-org/inoculate-or-reflect-camera-ready` dataset.

The seed-42 Gemma adapters must be available either under the configured Drive path or in the private dataset under `gemma4-12b/seed-42/{contaminated,crt_repair,strong_ip}/adapter/`. The notebook refuses to run with missing adapters rather than silently substituting a different seed or model.

In [ ]:
# Install the pinned stack used by the camera-ready mechanistic runner.
%pip install -q "transformers==5.13.1" "nnsight>=0.7,<0.8" "peft>=0.14" \
    "bitsandbytes>=0.43" "accelerate>=1.2" "huggingface_hub>=0.25" "safetensors"

In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

import torch
import transformers
import peft
import bitsandbytes
import nnsight

print("Python       ", sys.version.split()[0])
print("Torch        ", torch.__version__)
print("Transformers ", transformers.__version__)
print("PEFT         ", peft.__version__)
print("bitsandbytes ", bitsandbytes.__version__)
print("NNSight      ", nnsight.__version__)
print("CUDA         ", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Select a Colab A100 runtime before running the GPU cells.")
gpu_name = torch.cuda.get_device_name(0)
print("GPU          ", gpu_name)
if "A100" not in gpu_name:
    raise RuntimeError(f"This notebook is pinned to A100 behavior; found {gpu_name!r}.")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# Persistent locations. Change only the adapter dataset/path if your upload uses a different layout.
CAMERA_READY_DATASET = "ic-org/inoculate-or-reflect-camera-ready"
DRIVE_ROOT = Path("/content/drive/MyDrive/inoculate-or-reflect/camera-ready")
BUNDLE_ROOT = Path("/content/ior-camera-ready-remaining-bundle")
STAGED_ROOT = Path("/content/ior-camera-ready-remaining")
ADAPTER_DOWNLOAD_ROOT = Path("/content/ior-gemma-adapters")

GEMMA_ADAPTER_ROOT = DRIVE_ROOT / "adapters" / "gemma4-12b" / "seed-42"
GEMMA_MECH_OUT = DRIVE_ROOT / "mechanistic" / "gemma4-12b"
LONGCAP_OUT = DRIVE_ROOT / "longcap" / "gemma4-12b" / "seed-42" / "strong_ip"
MECH_LOG = DRIVE_ROOT / "logs" / "gemma4-12b-mechanistic.log"
LONGCAP_LOG = DRIVE_ROOT / "logs" / "gemma4-12b-longcap-strong-ip.log"

for path in (DRIVE_ROOT, GEMMA_ADAPTER_ROOT, GEMMA_MECH_OUT, LONGCAP_OUT, MECH_LOG.parent):
    path.mkdir(parents=True, exist_ok=True)
print("Drive root:          ", DRIVE_ROOT)
print("Gemma adapter root:  ", GEMMA_ADAPTER_ROOT)
print("Mechanistic output:  ", GEMMA_MECH_OUT)
print("Long-cap output:     ", LONGCAP_OUT)

In [ ]:
# Read the private token from Colab Secrets. Never paste a token into the notebook.
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    raise RuntimeError("Add a read-scoped HF_TOKEN in Colab Secrets, then rerun this cell.")
os.environ["HF_TOKEN"] = HF_TOKEN

from huggingface_hub import snapshot_download

# The bundle contains the exact runner, reserve prompts, and the sycophancy eval file used by long-cap.
if BUNDLE_ROOT.exists():
    shutil.rmtree(BUNDLE_ROOT)
snapshot_download(
    repo_id=CAMERA_READY_DATASET,
    repo_type="dataset",
    token=HF_TOKEN,
    local_dir=str(BUNDLE_ROOT),
    allow_patterns=["README.md", "artifact_manifest.json", "code/*", "data/**"],
)
for path in sorted(BUNDLE_ROOT.rglob("*")):
    if path.is_file():
        print(path.relative_to(BUNDLE_ROOT), path.stat().st_size, "bytes")

In [ ]:
# Stage the two standalone runners under the project-shaped root they expect.
def require_file(path: Path, purpose: str):
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {purpose}: {path}. Upload it to {CAMERA_READY_DATASET} before rerunning the setup cells."
        )

runner_src = BUNDLE_ROOT / "code" / "mechanistic_eval.py"
generate_src = BUNDLE_ROOT / "code" / "generate_eval.py"
reserve_src = BUNDLE_ROOT / "data" / "phase4_reserve.jsonl"
eval_src = BUNDLE_ROOT / "data" / "eval" / "eval_sycophancy.jsonl"
for path, purpose in (
    (runner_src, "mechanistic_eval.py"),
    (generate_src, "generate_eval.py"),
    (reserve_src, "phase4_reserve.jsonl"),
    (eval_src, "eval_sycophancy.jsonl"),
):
    require_file(path, purpose)

runner_dst = STAGED_ROOT / "experiments" / "camera_ready" / "mechanistic_eval.py"
generate_dst = STAGED_ROOT / "experiments" / "camera_ready" / "generate_eval.py"
reserve_dst = STAGED_ROOT / "data" / "gcd_sycophancy" / "phase4_reserve.jsonl"
eval_dst = STAGED_ROOT / "data" / "eval" / "eval_sycophancy.jsonl"
for dst in (runner_dst, generate_dst, reserve_dst, eval_dst):
    dst.parent.mkdir(parents=True, exist_ok=True)
for src, dst in ((runner_src, runner_dst), (generate_src, generate_dst), (reserve_src, reserve_dst), (eval_src, eval_dst)):
    shutil.copy2(src, dst)
print("Runner SHA-256: ", hashlib.sha256(runner_dst.read_bytes()).hexdigest())
print("Generator SHA-256:", hashlib.sha256(generate_dst.read_bytes()).hexdigest())
print("Staged root:      ", STAGED_ROOT)

In [ ]:
# Materialize the exact seed-42 Gemma adapters.
# If they are already on Drive, nothing is downloaded. Otherwise the notebook looks
# for the expected directory layout in the same private dataset.
def adapter_root_complete(root: Path) -> bool:
    required_arms = ("contaminated", "crt_repair", "strong_ip")
    for arm in required_arms:
        arm_dir = root / arm / "adapter"
        if not (arm_dir / "adapter_config.json").exists():
            return False
        if not any(arm_dir.glob("adapter_model.*")):
            return False
    return True

if not adapter_root_complete(GEMMA_ADAPTER_ROOT):
    if ADAPTER_DOWNLOAD_ROOT.exists():
        shutil.rmtree(ADAPTER_DOWNLOAD_ROOT)
    snapshot_download(
        repo_id=CAMERA_READY_DATASET,
        repo_type="dataset",
        token=HF_TOKEN,
        local_dir=str(ADAPTER_DOWNLOAD_ROOT),
        allow_patterns=["gemma4-12b/seed-42/**"],
    )
    candidates = []
    for config in ADAPTER_DOWNLOAD_ROOT.rglob("adapter_config.json"):
        candidate = config.parents[2]  # .../seed-42/<arm>/adapter/adapter_config.json
        if adapter_root_complete(candidate):
            candidates.append(candidate)
    if candidates:
        source = candidates[0]
        shutil.copytree(source, GEMMA_ADAPTER_ROOT, dirs_exist_ok=True)
        print("Copied adapters from private bundle:", source)

if not adapter_root_complete(GEMMA_ADAPTER_ROOT):
    raise RuntimeError(
        "Seed-42 Gemma adapters are unavailable. Upload contaminated/adapter, crt_repair/adapter, "
        "and strong_ip/adapter under gemma4-12b/seed-42/ in the private dataset, or copy them to " 
        f"{GEMMA_ADAPTER_ROOT}."
    )
for path in sorted(GEMMA_ADAPTER_ROOT.rglob("adapter_config.json")):
    print("Adapter:", path.relative_to(GEMMA_ADAPTER_ROOT))

In [ ]:
# Inspect existing checkpoints before launching either job.
def jsonl_count(path: Path) -> int:
    if not path.exists():
        return 0
    with path.open() as handle:
        return sum(1 for line in handle if line.strip())

mech_steering = GEMMA_MECH_OUT / "steering_results.jsonl"
mech_patch = GEMMA_MECH_OUT / "patch_results.jsonl"
longcap_rows = LONGCAP_OUT / "generations.jsonl"
print({
    "gemma_mechanistic_steering_rows": jsonl_count(mech_steering),
    "gemma_mechanistic_patch_rows": jsonl_count(mech_patch),
    "longcap_strong_ip_rows": jsonl_count(longcap_rows),
    "mechanistic_steering_remaining": max(0, 13600 - jsonl_count(mech_steering)),
    "mechanistic_patch_remaining": max(0, 2800 - jsonl_count(mech_patch)),
    "longcap_rows_remaining": max(0, 2700 - jsonl_count(longcap_rows)),
})

## Run Gemma mechanistic sweep

This is the predeclared seed-42 grid. It uses six relative depths, random-direction controls, zero-dose parity, and activation patching. The runner appends only unseen condition keys.

In [ ]:
runner = STAGED_ROOT / "experiments" / "camera_ready" / "mechanistic_eval.py"
cmd = [
    sys.executable, "-u", str(runner),
    "--model", "gemma4-12b",
    "--seed", "42",
    "--adapter-root", str(GEMMA_ADAPTER_ROOT),
    "--output-dir", str(GEMMA_MECH_OUT),
    "--batch-size", "8",
    "--max-new-tokens", "768",
]
print("$", " ".join(cmd))
env = os.environ.copy()
env["HF_TOKEN"] = HF_TOKEN
with MECH_LOG.open("a") as log:
    log.write("\n=== run " + " ".join(cmd) + " ===\n")
    process = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, env=env,
    )
    for line in process.stdout:
        print(line, end="")
        log.write(line)
    return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f"Gemma mechanistic runner exited with code {return_code}; checkpoints remain on Drive.")
print("Gemma mechanistic runner completed. Log:", MECH_LOG)

In [ ]:
# Validate the complete Gemma mechanistic artifact before moving on.
def read_jsonl(path: Path):
    with path.open() as handle:
        return [json.loads(line) for line in handle if line.strip()]

def unique_key_count(path: Path, fields):
    rows = read_jsonl(path)
    keys = {tuple(row.get(field) for field in fields) for row in rows}
    return len(rows), len(keys)

manifest_path = GEMMA_MECH_OUT / "manifest.json"
if not manifest_path.exists():
    raise RuntimeError("No Gemma mechanistic manifest was written; rerun the mechanistic cell to resume.")
manifest = json.loads(manifest_path.read_text())
steering_rows, steering_keys = unique_key_count(
    GEMMA_MECH_OUT / "steering_results.jsonl",
    ("analysis", "arm", "direction", "layer", "dose", "sign", "id"),
)
patch_rows, patch_keys = unique_key_count(
    GEMMA_MECH_OUT / "patch_results.jsonl",
    ("analysis", "source", "destination", "layers_key", "amount", "shuffled", "id"),
)
assert manifest["status"] == "complete"
assert steering_rows == steering_keys == 13600
assert patch_rows == patch_keys == 2800
print(json.dumps({
    "status": manifest["status"],
    "model": manifest["model"],
    "steering_rows": steering_rows,
    "patch_rows": patch_rows,
    "primary_layer": manifest["primary_layer"],
    "relative_layers": manifest["relative_layers"],
}, indent=2))

## Finish the Gemma long-cap Strong-IP panel

The target is 2,700 rows: 300 sycophancy prompts × 3 samples × baseline, generic restoration, and exact-IP restoration. Existing rows on Drive are reused automatically.

In [ ]:
generator = STAGED_ROOT / "experiments" / "camera_ready" / "generate_eval.py"
strong_ip_adapter = GEMMA_ADAPTER_ROOT / "strong_ip" / "adapter"
cmd = [
    sys.executable, "-u", str(generator),
    "--model", "gemma4-12b",
    "--seed", "42",
    "--arm", "strong_ip",
    "--adapter", str(strong_ip_adapter),
    "--output-dir", str(LONGCAP_OUT),
    "--batch-size", "16",
    "--max-new-tokens", "1024",
    "--sycophancy-only",
]
print("$", " ".join(cmd))
env = os.environ.copy()
env["HF_TOKEN"] = HF_TOKEN
with LONGCAP_LOG.open("a") as log:
    log.write("\n=== run " + " ".join(cmd) + " ===\n")
    process = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, env=env,
    )
    for line in process.stdout:
        print(line, end="")
        log.write(line)
    return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f"Long-cap generator exited with code {return_code}; checkpoints remain on Drive.")
print("Long-cap generator completed. Log:", LONGCAP_LOG)

In [ ]:
# Validate the long-cap manifest and write a compact completion record on Drive.
longcap_manifest = LONGCAP_OUT / "generation_manifest.json"
if not longcap_manifest.exists():
    raise RuntimeError("No long-cap manifest was written; rerun the generation cell to resume.")
longcap = json.loads(longcap_manifest.read_text())
assert longcap["status"] == "complete"
assert longcap["model_key"] == "gemma4-12b"
assert longcap["seed"] == 42
assert longcap["arm"] == "strong_ip"
assert longcap["sycophancy_only"] is True
assert longcap["n_rows"] == 2700
summary = {
    "gemma_mechanistic": {
        "manifest": str(GEMMA_MECH_OUT / "manifest.json"),
        "steering_rows": 13600,
        "patch_rows": 2800,
    },
    "gemma_longcap_strong_ip": {
        "manifest": str(longcap_manifest),
        "rows": longcap["n_rows"],
        "max_new_tokens": longcap["max_new_tokens_by_eval"],
    },
    "completed_at_unix": time.time(),
}
summary_path = DRIVE_ROOT / "remaining_gpu_experiments_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
print("Summary written to", summary_path)

## Export for local grading

Keep the full directories on Drive. When the run is complete, either sync those directories back into `outputs/camera_ready/` or download an archive. The local next steps are frozen mechanistic grading, long-cap grading, figure regeneration, and paper integration.

In [ ]:
# Optional: make one archive containing only the two completed output directories.
export_root = Path("/content/camera-ready-remaining")
if export_root.exists():
    shutil.rmtree(export_root)
(export_root / "mechanistic").parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(GEMMA_MECH_OUT, export_root / "mechanistic")
shutil.copytree(LONGCAP_OUT, export_root / "longcap_strong_ip")
archive = shutil.make_archive("/content/camera-ready-remaining", "zip", root_dir=export_root)
print(archive)
print("The same files remain on Drive at", DRIVE_ROOT)